# 01 — Carregamento e inspeção inicial do SSVEP Benchmark Dataset

**Objetivo:** organizar o ambiente, localizar os arquivos extraídos e inspecionar a estrutura do arquivo EEG `S1.mat`, sem modificar os dados originais.

**Dataset:** Benchmark Dataset para SSVEP-BCI, disponibilizado pelo grupo de pesquisa em BCI da Universidade Tsinghua.

> Esta é uma versão inicial. Os nomes e as dimensões internas das variáveis serão confirmados após executar as células de inspeção.

## 1. Importar bibliotecas

Utilizamos bibliotecas comuns para análise científica em Python.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

print("Bibliotecas importadas com sucesso.")

## 2. Definir o caminho dos dados

Estrutura esperada:

```text
ssvep-benchmark-analysis/
├── notebooks/
│   └── 01_load_dataset_atualizado.ipynb
└── data/
    ├── raw/
    │   ├── S1.mat
    │   ├── Freq_Phase.mat
    │   ├── Sub_info.txt
    │   └── 64-channels.loc
    └── processed/
```

Como o notebook fica em `notebooks/`, `../data/raw` aponta para a pasta dos dados. Se o notebook estiver em outra pasta, ajuste `DATA_DIR`.

In [ ]:
DATA_DIR = Path("../data/raw")

print("Pasta de dados:", DATA_DIR.resolve())
print("Pasta existe:", DATA_DIR.exists())

## 3. Listar os arquivos disponíveis

A célula lista os arquivos e seus tamanhos. Ela não abre nem altera os dados.

In [ ]:
if not DATA_DIR.exists():
    print("Pasta não encontrada. Confira a estrutura do projeto e o valor de DATA_DIR.")
else:
    arquivos = sorted(p for p in DATA_DIR.iterdir() if p.is_file())

    if not arquivos:
        print("A pasta existe, mas está vazia.")
    else:
        for arquivo in arquivos:
            tamanho_mb = arquivo.stat().st_size / (1024 ** 2)
            print(f"{arquivo.name} — {tamanho_mb:.2f} MB")

## 4. Localizar e carregar `S1.mat`

`scipy.io.loadmat` lê arquivos MAT compatíveis com SciPy. Se o arquivo tiver sido salvo no formato MATLAB v7.3 (baseado em HDF5), poderá ser necessário usar `h5py`; nesse caso, confira a mensagem de erro antes de tentar converter o arquivo.

In [ ]:
arquivo_eeg = DATA_DIR / "S1.mat"

if not arquivo_eeg.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo_eeg.resolve()}\n"
        "Confirme se S1.mat está em data/raw e se DATA_DIR está correto."
    )

dados = loadmat(arquivo_eeg)
print("Arquivo carregado:", arquivo_eeg.name)

## 5. Inspecionar as variáveis internas

A célula mostra as variáveis que não são metadados internos do MATLAB, incluindo formato, tipo e número de elementos.

In [1]:
variaveis = []

for chave, valor in dados.items():
    if chave.startswith("__"):
        continue

    if isinstance(valor, np.ndarray):
        variaveis.append({
            "variavel": chave,
            "dimensoes": valor.shape,
            "tipo": str(valor.dtype),
            "numero_elementos": valor.size
        })
    else:
        variaveis.append({
            "variavel": chave,
            "dimensoes": None,
            "tipo": type(valor).__name__,
            "numero_elementos": None
        })

resumo_variaveis = pd.DataFrame(variaveis)
resumo_variaveis

NameError: name 'dados' is not defined

## 6. Próximas etapas

Depois de executar o notebook, guarde a tabela de variáveis e dimensões. Ela será usada para identificar a matriz de sinais EEG e confirmar a organização por canal, amostra, alvo e bloco.

Na próxima versão, poderemos inspecionar os metadados de frequência/fase, calcular estatísticas descritivas e criar gráficos iniciais.

**Ainda não há filtragem, normalização, segmentação ou transformação dos sinais.** Preserve os arquivos brutos para manter a rastreabilidade.

## 7. Interpretar as dimensões dos sinais

A variável `data` foi identificada com dimensões `(64, 1500, 40, 6)`. Para este dataset, interpretamos os eixos como:

1. canais EEG;
2. amostras temporais por ensaio;
3. alvos visuais;
4. blocos experimentais.

A frequência de amostragem informada para o Benchmark Dataset é de 250 Hz. Assim, 1500 amostras correspondem a 6 segundos por ensaio. Nesta etapa, apenas inspecionamos os dados; não aplicamos filtros nem alteramos a matriz original.

In [2]:
# Conferir a estrutura esperada antes de prosseguir
eeg = dados["data"]

print("Formato:", eeg.shape)
print("Tipo:", eeg.dtype)
print("Número total de valores:", eeg.size)
print(f"Tamanho aproximado da matriz: {eeg.nbytes / (1024**2):.2f} MiB")

assert eeg.ndim == 4, "Esperava-se uma matriz de quatro dimensões."
assert eeg.shape == (64, 1500, 40, 6), (
    "As dimensões não correspondem à estrutura esperada. "
    "Confira o arquivo e a documentação antes de continuar."
)

NameError: name 'dados' is not defined

## 8. Estatísticas descritivas gerais

Vamos verificar valores ausentes, valores não finitos, amplitude e estatísticas globais. Como os dados podem ser grandes, as operações são realizadas diretamente sobre a matriz NumPy.

In [ ]:
total = eeg.size
nao_finitos = int(np.size(eeg) - np.count_nonzero(np.isfinite(eeg)))

estatisticas = pd.DataFrame({
    "medida": [
        "mínimo",
        "máximo",
        "média",
        "desvio padrão",
        "mediana",
        "valores não finitos"
    ],
    "valor": [
        np.nanmin(eeg),
        np.nanmax(eeg),
        np.nanmean(eeg),
        np.nanstd(eeg),
        np.nanmedian(eeg),
        nao_finitos
    ]
})

estatisticas

## 9. Visualizar um sinal EEG

O gráfico abaixo mostra um canal, um alvo e um bloco como exemplo. A escolha é apenas ilustrativa; não significa que esse canal ou alvo seja mais importante. O eixo temporal é calculado usando a frequência de amostragem de 250 Hz.

In [ ]:
FREQUENCIA_AMOSTRAGEM = 250  # Hz

canal = 0
alvo = 0
bloco = 0

sinal = eeg[canal, :, alvo, bloco]
tempo = np.arange(sinal.size) / FREQUENCIA_AMOSTRAGEM

plt.figure(figsize=(12, 4))
plt.plot(tempo, sinal, linewidth=0.8)
plt.title(f"EEG — canal {canal + 1}, alvo {alvo + 1}, bloco {bloco + 1}")
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude (unidade conforme documentação do dataset)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Comparar a amplitude entre canais

A próxima visualização resume a amplitude média quadrática (RMS) de cada canal, calculada sobre todas as amostras, alvos e blocos. Essa medida descreve a magnitude do sinal; não deve ser interpretada isoladamente como atenção, qualidade clínica ou desempenho de classificação.

In [ ]:
rms_por_canal = np.sqrt(np.mean(eeg ** 2, axis=(1, 2, 3)))

plt.figure(figsize=(12, 4))
plt.bar(np.arange(1, eeg.shape[0] + 1), rms_por_canal)
plt.title("Amplitude RMS por canal")
plt.xlabel("Canal")
plt.ylabel("RMS (unidade conforme documentação)")
plt.xticks(np.arange(1, eeg.shape[0] + 1, 4))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

pd.DataFrame({
    "canal": np.arange(1, eeg.shape[0] + 1),
    "rms": rms_por_canal
}).sort_values("rms", ascending=False).head(10)

## 11. Verificar valores não finitos por canal

Esta verificação ajuda a detectar `NaN` e infinitos. Ela não identifica todos os possíveis problemas de qualidade do sinal, mas é uma checagem inicial útil.

In [ ]:
nao_finitos_por_canal = np.sum(~np.isfinite(eeg), axis=(1, 2, 3))

pd.DataFrame({
    "canal": np.arange(1, eeg.shape[0] + 1),
    "valores_nao_finitos": nao_finitos_por_canal
})

## 12. Registro das conclusões iniciais

Após executar as células, registre no relatório ou README:

- formato confirmado da matriz e interpretação dos eixos;
- frequência de amostragem utilizada para construir o eixo temporal;
- estatísticas globais e presença/ausência de valores não finitos;
- observações visuais sobre o sinal e diferenças de amplitude entre canais.

**Limitações:** esta é uma análise exploratória inicial de um participante (`S1`). Não permite generalizar resultados para todos os participantes. Antes de interpretar unidades, canais específicos ou classes, consulte o `Readme.txt`, o arquivo de localização dos canais e os metadados oficiais.

**Ainda não foi aplicado pré-processamento:** os sinais originais permanecem inalterados.